# Pipeline outputs analysis

This notebook loads **all pipeline output CSVs**, explains what each metric means, and visualizes results with **scatter plots**, **heatmaps**, **decision-boundary style charts**, and **line/bar plots**.

**Sections:**
1. **Probes** — Where in the model (layer × loc) do emotion probes work best?
2. **Circuit evidence** — Single-best vs top-k fusion
3. **Emotion circuit** — Which layers form the circuit per emotion
4. **Appraisal structure** — Emotions in appraisal space (scatter, decision boundary)
5. **Steering** — Appraisal vs emotion steering (success vs strength)
6. **Probe robustness** — Dataset variant comparison

Run from **repo root** with `PYTHONPATH` set.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Repo root = parent of pipeline
REPO_ROOT = Path.cwd() if (Path.cwd() / "pipeline").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline.config import DEFAULT_MODEL_ID, get_model_output_dir, get_probe_paths, get_circuit_dir, get_appraisal_structure_dir, get_probe_robustness_dir, get_steering_dir

MODEL_ID = DEFAULT_MODEL_ID
OUT = get_model_output_dir(MODEL_ID)
print(f"Model: {MODEL_ID}")
print(f"Outputs root: {OUT}")

---
## 1. Probe performance (layer × loc)

**What it is:** For each (layer, location) we train a small classifier to predict one emotion vs rest from hidden states. **test_roc_auc** measures how well the probe separates that emotion from others (1.0 = perfect, 0.5 = random).

**Why it matters:** High ROC-AUC at certain layers/locs means "this emotion is linearly decodable here" — the representation is **readable** at that point in the model.

In [ ]:
# Load probe summary (01_probes or input_probes)
paths = get_probe_paths(MODEL_ID)
probe_summary_path = paths.probe_summary_csv
ps = None
if probe_summary_path.exists():
    ps = pd.read_csv(probe_summary_path)
else:
    print(f"Probe summary not found: {probe_summary_path}. Run train_probes first. Skipping probe plots.")
if ps is not None:
    ps["layer"] = ps["layer"].astype(int)
    ps["loc"] = ps["loc"].astype(int)
    by_layer_loc = ps.groupby(["layer", "loc"])["test_roc_auc"].mean().reset_index()
    pivot = by_layer_loc.pivot(index="layer", columns="loc", values="test_roc_auc")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=1.0)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Location index")
    ax.set_ylabel("Layer index")
    ax.set_title("Mean probe ROC-AUC by layer × loc\n(green = better emotion decoding)")
    plt.colorbar(im, ax=ax, label="Mean test_roc_auc")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax = axes[1]
    sizes = (by_layer_loc["test_roc_auc"] - 0.5) * 200 + 20
    sc = ax.scatter(by_layer_loc["loc"], by_layer_loc["layer"], s=sizes, c=by_layer_loc["test_roc_auc"], cmap="RdYlGn", vmin=0.5, vmax=1.0, alpha=0.8)
    ax.set_xlabel("Location")
    ax.set_ylabel("Layer")
    ax.set_title("Probe performance: (layer, loc)\n(bubble size & color = ROC-AUC)")
    plt.colorbar(sc, ax=ax, label="test_roc_auc")
    plt.tight_layout()
    plt.show()

In [ ]:
if ps is not None:
    # Scatter: test_accuracy vs test_roc_auc (each point = one probe)
    # "Decision boundary" = ROC-AUC > 0.8 (green zone) vs below (red zone)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.axhspan(0.8, 1.0, alpha=0.2, color="green", label="ROC-AUC ≥ 0.8 (good)")
    ax.axhspan(0.5, 0.8, alpha=0.2, color="orange")
    ax.scatter(ps["test_accuracy"], ps["test_roc_auc"], alpha=0.4, s=15, c="navy")
    ax.axhline(0.8, color="green", linestyle="--", linewidth=1, label="Threshold 0.8")
    ax.set_xlabel("Test accuracy")
    ax.set_ylabel("Test ROC-AUC")
    ax.set_title("Probe performance: accuracy vs ROC-AUC\n(green zone = strong decoding)")
    ax.legend(loc="lower right")
    ax.set_ylim(0.45, 1.02)
    plt.tight_layout()
    plt.show()

---
## 2. Circuit evidence (single-best vs top-k fusion)

**What it is:** We compare two ways to classify emotion from hidden states:
- **single_best:** Use the single best (layer, loc) probe.
- **topk_fusion:** Average probe logits over the top-k (layer, loc) pairs, then predict.

**Why it matters:** If top-k fusion beats single-best, emotion is represented across **multiple** layers/locs — a "circuit" rather than one spot. The **decision boundary** for emotion is then a combination of several linear boundaries.

In [ ]:
circuit_csv = get_circuit_dir(MODEL_ID) / "circuit_evidence_classification.csv"
if circuit_csv.exists():
    ce = pd.read_csv(circuit_csv)
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(ce))
    w = 0.35
    ax.bar(x - w/2, ce["accuracy"], width=w, label="Accuracy", color="steelblue")
    if "roc_auc" in ce.columns and ce["roc_auc"].notna().any():
        ax.bar(x + w/2, ce["roc_auc"], width=w, label="Macro ROC-AUC", color="coral")
    ax.set_xticks(x)
    ax.set_xticklabels(ce["method"])
    ax.set_ylabel("Score")
    ax.set_title("Circuit evidence: single-best vs top-k fusion\n(higher = better classification)")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
    display(ce)
else:
    print("circuit_evidence_classification.csv not found. Run circuit_evidence first.")

---
## 3. Emotion circuit (which layers per emotion?)

**What it is:** For each emotion, we take the **top-k layers** by mean ROC-AUC (across locs). That list is the "circuit" for that emotion. The heatmap below shows **emotion × layer**: 1 if the layer is in that emotion's circuit, 0 otherwise.

**Why it matters:** You see which layers participate in which emotions. Overlap between emotions suggests shared representation; unique layers suggest emotion-specific processing.

In [ ]:
circuits_path = get_circuit_dir(MODEL_ID) / "circuits.json"
if circuits_path.exists():
    with open(circuits_path) as f:
        circuits = json.load(f)
    emotions = list(circuits.keys())
    layers = sorted(set(l for em in circuits for l in circuits[em]))
    # Build matrix: emotion × layer → 1 if in circuit else 0
    mat = np.zeros((len(emotions), len(layers)))
    for i, em in enumerate(emotions):
        for L in circuits[em]:
            j = layers.index(L)
            mat[i, j] = 1
    fig, ax = plt.subplots(figsize=(max(8, len(layers)*0.5), max(5, len(emotions)*0.4)))
    ax.imshow(mat, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(layers)))
    ax.set_xticklabels(layers)
    ax.set_yticks(range(len(emotions)))
    ax.set_yticklabels(emotions)
    ax.set_xlabel("Layer (in circuit = 1)")
    ax.set_ylabel("Emotion")
    ax.set_title("Emotion circuit: which layers are top-k for each emotion?")
    plt.tight_layout()
    plt.show()
else:
    print("circuits.json not found. Run phase1_circuits first.")

---
## 4. Appraisal structure (emotions in appraisal space)

**What it is:** From dataset labels we compute mean appraisal ratings (pleasantness, control, etc.) per emotion, then z-score across emotions. So each emotion is a point in **appraisal space**.

**Decision-boundary view:** We plot **pleasantness vs unpleasantness**. Roughly: positive pleasantness + negative unpleasantness → positive emotions (e.g. joy); negative pleasantness + positive unpleasantness → negative emotions (e.g. anger). The diagonal separates "positive" from "negative" appraisal regions.

In [ ]:
appraisal_csv = get_appraisal_structure_dir(MODEL_ID) / "appraisal_zscore_by_emotion.csv"
if appraisal_csv.exists():
    zscore = pd.read_csv(appraisal_csv, index_col=0)
    emotions = list(zscore.index)
    # Scatter: pleasantness vs unpleasantness with decision-boundary style
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    x = zscore["pleasantness"]
    y = zscore["unpleasantness"]
    colors = ["green" if x[e] > 0 and y[e] < 0 else "red" if x[e] < 0 and y[e] > 0 else "gray" for e in emotions]
    for i, e in enumerate(emotions):
        ax.scatter(x.iloc[i], y.iloc[i], c=colors[i], s=80, alpha=0.8, edgecolors="black")
        ax.annotate(e, (x.iloc[i], y.iloc[i]), xytext=(5, 5), textcoords="offset points", fontsize=8)
    ax.axhline(0, color="k", linewidth=0.5)
    ax.axvline(0, color="k", linewidth=0.5)
    # Diagonal "decision boundary": pleasantness = - unpleasantness
    lim = max(abs(x).max(), abs(y).max()) * 1.1
    ax.plot([-lim, lim], [lim, -lim], "b--", alpha=0.5, label="Pleasant ≈ -Unpleasant")
    ax.set_xlabel("Pleasantness (z-score)")
    ax.set_ylabel("Unpleasantness (z-score)")
    ax.set_title("Emotions in appraisal space\n(green ≈ positive, red ≈ negative)")
    ax.legend()
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")

    # Heatmap: emotion × appraisal dimension
    ax = axes[1]
    im = ax.imshow(zscore.values, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set_yticks(range(len(emotions)))
    ax.set_yticklabels(emotions)
    ax.set_xticks(range(len(zscore.columns)))
    ax.set_xticklabels(zscore.columns, rotation=45, ha="right")
    ax.set_title("Appraisal z-scores by emotion")
    plt.colorbar(im, ax=ax, label="z-score")
    plt.tight_layout()
    plt.show()
else:
    print("appraisal_zscore_by_emotion.csv not found. Run appraisal_structure first.")

---
## 5. Steering (appraisal vs emotion steering)

**What it is:** We take **anger** scenarios and add a steering vector to hidden states (either appraisal-based or emotion-probe-based) to push the representation toward **joy**. We measure **success rate** (fraction classified as joy) and **mean target logit** (how strong the "joy" signal is) as we increase steering strength.

**Decision-boundary idea:** Success rate is the fraction of samples that cross the probe's **decision boundary** (e.g. joy logit > 0.5). The curves show how many samples cross that boundary as strength increases.

In [ ]:
steering_dir = get_steering_dir(MODEL_ID)
curves_path = steering_dir / "steering_curves.csv"
benchmark_path = steering_dir / "steering_benchmark.csv"
if curves_path.exists():
    curves = pd.read_csv(curves_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for method in curves["method"].unique():
        sub = curves[curves["method"] == method]
        axes[0].plot(sub["strength"], sub["success_rate"], "o-", label=method)
        axes[1].plot(sub["strength"], sub["mean_target_logit"], "o-", label=method)
    axes[0].axhline(0.5, color="gray", linestyle="--", alpha=0.7)
    axes[0].set_xlabel("Steering strength")
    axes[0].set_ylabel("Success rate (anger→joy)")
    axes[0].set_title("Fraction classified as joy\n(above 0.5 = majority cross decision boundary)")
    axes[0].legend()
    axes[0].set_ylim(-0.05, 1.05)
    axes[1].set_xlabel("Steering strength")
    axes[1].set_ylabel("Mean joy logit")
    axes[1].set_title("Mean joy probe output")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
if benchmark_path.exists():
    bench = pd.read_csv(benchmark_path)
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(bench))
    w = 0.35
    ax.bar(x - w/2, bench["success_rate"], width=w, label="Success rate", color="steelblue")
    ax.bar(x + w/2, bench["mean_delta_target_logit"], width=w, label="Mean Δ joy logit", color="coral")
    ax.set_xticks(x)
    ax.set_xticklabels(bench["method"])
    ax.set_ylabel("Score")
    ax.set_title("Steering benchmark at strength 1.0")
    ax.legend()
    plt.tight_layout()
    plt.show()
    display(bench)
if not curves_path.exists() and not benchmark_path.exists():
    print("Steering CSVs not found. Run steering_benchmark first.")

---
## 6. Probe robustness (dataset variants)

**What it is:** We train probes on different **dataset variants** (e.g. prompted only vs prompted+unprompted). Each variant uses a different way to turn scenarios into text. We compare **mean validation ROC-AUC** and **accuracy** across variants.

**Why it matters:** If performance is similar across variants, the probes are **robust** to how the text is phrased. If one variant wins, that formulation may be better for decoding emotion.

In [ ]:
robust_path = get_probe_robustness_dir(MODEL_ID) / "probe_robustness_comparison.csv"
if robust_path.exists():
    rob = pd.read_csv(robust_path)
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(rob))
    w = 0.35
    ax.bar(x - w/2, rob["mean_val_roc_auc"], width=w, label="Mean val ROC-AUC", color="steelblue")
    ax.bar(x + w/2, rob["mean_val_accuracy"], width=w, label="Mean val accuracy", color="coral")
    ax.set_xticks(x)
    ax.set_xticklabels(rob["variant"], rotation=15, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Probe robustness: performance by dataset variant")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
    display(rob)
else:
    print("probe_robustness_comparison.csv not found. Run probe_training_robustness with quick probes.")

---
## 7. Baseline classification & cluster mapping

**Baseline:** Emotion classification accuracy and macro ROC-AUC when using the **single best (layer, loc)** probe on a sample.  
**Cluster mapping:** K-means on probe logits groups samples; each cluster is labeled by its **majority emotion**. Shows how well probe-derived features cluster by emotion.

In [ ]:
base_path = get_appraisal_structure_dir(MODEL_ID) / "baseline_metrics.csv"
cluster_path = get_appraisal_structure_dir(MODEL_ID) / "cluster_emotion_mapping.csv"
if base_path.exists():
    base = pd.read_csv(base_path)
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.bar(base["metric"], base["value"], color=["steelblue", "coral"])
    ax.set_ylabel("Score")
    ax.set_title("Baseline emotion classification (best probe)")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
    display(base)
if cluster_path.exists():
    clu = pd.read_csv(cluster_path)
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [f"C{r['cluster']} ({r['majority_emotion']})" for _, r in clu.iterrows()]
    ax.barh(range(len(clu)), clu["size"], color="steelblue", alpha=0.8)
    ax.set_yticks(range(len(clu)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("Cluster size")
    ax.set_title("K-means clusters on probe logits (label = majority emotion)")
    plt.tight_layout()
    plt.show()
    display(clu)

---
## Summary

| Output | Meaning |
|--------|--------|
| **Probe heatmap** | Where in the model (layer × loc) emotion is linearly decodable. |
| **Circuit evidence** | Combining several layer/loc probes (top-k) improves over one — emotion is a "circuit". |
| **Circuit heatmap** | Which layers are in the top-k for each emotion. |
| **Appraisal scatter** | Emotions live in appraisal space; pleasantness vs unpleasantness separates positive/negative. |
| **Steering curves** | How success rate and joy logit grow with steering strength (crossing the decision boundary). |
| **Probe robustness** | Whether probe performance is stable across text variants. |
| **Baseline / clusters** | Single-probe accuracy and how probe logits cluster by emotion. |